# pycograd — when `vmap` earns its keep

[`vmap`](https://jax.readthedocs.io/en/latest/_autosummary/jax.vmap.html) turns a
function written for **one example** into one that runs over a whole batch in a single
vectorized pass. A fair question is *when you actually need it.* For a plain feedforward
net — matmuls and elementwise ops — you don't: numpy already broadcasts those over the
leading axis, so `forward(X)` on the full `(N, …)` dataset **is** the batched forward,
and wrapping it in `vmap` changes nothing.

`vmap` earns its keep in two situations, and this notebook is built around them:

1. **Per-sample gradients.** A single full-batch backward returns *one* gradient — the
   batch **mean** — having summed the per-example contributions away.
   `vmap(grad(per_example_loss))` keeps them separate: one gradient per example. That is
   what gradient clipping, DP-SGD, and influence analysis need, and broadcasting alone
   cannot recover it.
2. **Per-example computations that don't broadcast.** A self-attention layer's
   $QK^\top$ is written for one `length × d` sequence; applied naively to a
   `(B, length, d)` batch it would mix examples together. `vmap` maps the per-sequence
   forward over the batch correctly — collapsing what would otherwise be an explicit
   Python loop over examples.

`vmap` is one level of pycograd's trace-level interpreter stack, so it **composes** with
`grad` both ways: `grad` *through* a `vmap`'d forward, and `vmap` *over* `grad` for the
per-sample case above.

**Requires** the `pipescript` extra: `pip install pycograd[notebook]`.

In [1]:
%load_ext pycograd

## The primitive: `grad`

Underneath the DSL, `grad` / `value_and_grad` differentiate an ordinary numpy
function — the array argument is lifted onto the tape for you.

In [2]:
import numpy as np
from pycograd import grad

def f(x):
    return np.sum(np.sin(x * x))

x = np.array([0.5, 1.0, 1.5])
(gx,) = grad(f)(x)
print("autodiff :", np.round(gx, 4))
print("analytic :", np.round(2 * x * np.cos(x * x), 4))

autodiff : [ 0.9689  1.0806 -1.8845]
analytic : [ 0.9689  1.0806 -1.8845]


## The primitive: `vmap`

`vmap(g)` turns a function written for a **single** input into one that runs over a
whole batch — vectorized (one pass), not looped. Here `g` maps a length-3 vector to a
scalar; `vmap(g)` maps a `(B, 3)` batch to a `(B,)` vector of scalars, matching a
Python loop exactly but without one. Use `in_axes` to mark which arguments are
batched (an int axis) versus shared across the batch (`None`).

In [3]:
from pycograd import vmap

def g(v):                      # one length-3 vector -> one scalar
    return np.sum(np.sin(v * v))

batch = np.array([[0.5, 1.0, 1.5],
                  [0.2, 0.4, 0.6],
                  [1.0, 1.0, 1.0]])

vectorized = vmap(g)(batch)                      # one batched pass
looped     = np.array([g(v) for v in batch])     # the Python-loop oracle
print("vmap   :", np.round(vectorized, 4))
print("loop   :", np.round(looped, 4))

# vmap composes with grad: per-sample gradients, stacked over the batch.
per_sample = vmap(grad(g))(batch)
print("per-sample grads:\n", np.round(per_sample, 4))

vmap   : [1.8669 0.5516 2.5244]
loop   : [1.8669 0.5516 2.5244]
per-sample grads:
 [[[ 0.9689  1.0806 -1.8845]
  [ 0.3997  0.7898  1.1231]
  [ 1.0806  1.0806  1.0806]]]


## Setup: data and a loss

There is no op library to import for the model — pycograd differentiates raw numpy. We
make a 3-class blob dataset and reach for the library's first-class `cross_entropy` (mean
soft-target softmax CE *from logits*). The same op serves both views we need: on an
`(N, classes)` block it's the batch-mean loss, and on a **single** example's `(classes,)`
logits it's that one example's loss — the per-example loss the per-sample-gradient pillar
differentiates. A linear layer is just `$ @ w + b` and `relu` is `np.maximum(0.0, $)`.

In [4]:
from pycograd import cross_entropy   # batch-mean on (N, C); single-example on (C,)

rng = np.random.default_rng(0)
m = 40
centers = np.array([[2.0, 2.0], [-2.0, 2.0], [0.0, -2.5]])
X = np.vstack([rng.normal(c, 0.5, (m, 2)) for c in centers])
labels = np.repeat(np.arange(3), m)
Y = np.eye(3)[labels]   # one-hot, 3 classes

## Why a feedforward `vmap` is a no-op

Before the cases where `vmap` matters, it's worth seeing the case where it doesn't. A
linear (or MLP) forward is matmul + elementwise ops, and numpy already batches those
over the leading axis. So `vmap(forward)(X)` computes the **exact same thing** as calling
`forward` on the whole dataset — bit for bit. For these models you would just write
`forward(X)`; reaching for `vmap` adds a trace level and buys nothing.

In [5]:
w = rng.standard_normal((2, 3)); b = rng.standard_normal(3)
forward = $ |> $ @ w + b                # a one-point pipeline: (2,) -> (3,) logits

direct = forward(X)                      # numpy broadcasts (N,2) -> (N,3): already batched
mapped = vmap(forward)(X)                # map the per-example forward over the batch
print("shapes      :", direct.shape, mapped.shape)
print("bit-identical:", np.array_equal(direct, mapped))   # vmap added nothing

shapes      : (120, 3) (120, 3)
bit-identical: True


## Pillar 1 — per-sample gradients

Here is something broadcasting **cannot** give you. The weights are shared across the
batch, so a single full-batch backward returns one gradient — the batch mean — with the
per-example contributions already summed together. To get them back, write the loss for
**one** example and map `grad` over the batch with `vmap`: weights shared (`in_axes=None`),
data mapped (`in_axes=0`). `grad` differentiates all of `per_example_loss`'s arguments;
we keep the two w.r.t. the weights.

As a sanity check, the **mean** of the per-sample gradients is exactly the full-batch
gradient — the per-sample view simply refuses to throw the spread away.

In [6]:
def per_example_loss(w, b, x, y):       # ONE (2,) point + ONE label -> scalar
    return x |> $ @ w + b |> cross_entropy($, y)

w = np.zeros((2, 3)); b = np.zeros(3)
gw, gb, _, _ = vmap(grad(per_example_loss), in_axes=(None, None, 0, 0))(w, b, X, Y)
print("per-sample gw:", gw.shape, " gb:", gb.shape)   # (N, 2, 3) and (N, 3)

# the batch-mean of the per-sample grads is exactly the full-batch gradient
def batch_loss(w, b, X, Y):
    return X |> $ @ w + b |> cross_entropy($, Y)
fgw, fgb, _, _ = grad(batch_loss)(w, b, X, Y)
print("mean(per-sample) == full-batch:",
      np.allclose(gw.mean(0), fgw), np.allclose(gb.mean(0), fgb))

per-sample gw: (120, 2, 3)  gb: (120, 3)
mean(per-sample) == full-batch: True True


### Per-example gradient clipping

Per-sample gradients are exactly what **gradient clipping** (the core of DP-SGD) needs:
bound *each example's* gradient norm before averaging, so no single example can dominate
the step. That per-example clip is impossible from one summed backward — you only have
the average, not the pieces. The same blob classifier trains fine under it.

In [7]:
clip = 1.0
w = np.zeros((2, 3)); b = np.zeros(3)
first = last = None
for _ in range(200):
    gw, gb, _, _ = vmap(grad(per_example_loss), in_axes=(None, None, 0, 0))(w, b, X, Y)
    norms = np.sqrt((gw ** 2).sum((1, 2)) + (gb ** 2).sum(1))   # one norm per example
    scale = np.minimum(1.0, clip / (norms + 1e-12))             # per-example clip factor
    gw_c = (gw * scale[:, None, None]).mean(0)                  # clip, THEN average
    gb_c = (gb * scale[:, None]).mean(0)
    value = float(X @ w + b |> cross_entropy($, Y))
    first = value if first is None else first
    last = value
    w = w - 0.5 * gw_c; b = b - 0.5 * gb_c

print("clipped-SGD loss %.4f -> %.4f" % (first, last))
print("train accuracy:", np.mean(np.argmax(X @ w + b, axis=1) == labels))

clipped-SGD loss 1.0986 -> 0.0040
train accuracy: 1.0


## A small layer toolkit for the sequence models

The Pillar 2 models reuse a few **first-class ops imported straight from `pycograd`** —
`relu`, `layer_norm`, and scaled dot-product `attention`. They are written for a single
`length × d` sequence and `vmap` maps them over the batch with no attention-specific rule
(differentiated whenever a tracer flows through — a `Var` under `grad`, or a `vmap` batch
tracer here). The training loop is `train(...)` from `pycograd` — the same loop
as before, which also drives a *stateful* optimizer like Adam in place. (`cross_entropy`
comes from the setup cell.)

In [8]:
from pycograd import Adam, cosine_decay, layer_norm, relu
from pycograd import scaled_dot_product_attention as attention
from pycograd import train

## Pillar 2 — a self-attention sequence classifier

Now the inputs are short **sequences** (each a `length × d` matrix), and the label
depends on the tokens jointly. A self-attention layer lets every position attend to every
other — $\mathrm{softmax}(QK^\top/\sqrt{d})\,V$ — before we mean-pool over positions and
read out a class.

**This is where `vmap` earns its keep.** That $QK^\top$ is written for one `length × d`
sequence; on a stacked `(B, length, d)` batch it would entangle examples. So `forward1`
is written for a **single** sequence and `vmap(forward1)` maps it over the batch in one
pass — replacing the explicit Python `for`-loop over examples that the un-batched demo
needed.

In [9]:
# synthetic sequences: two classes separated by a per-token mean shift
def make_sequences(n_per=12, L=4, d=8, seed=7):
    g = np.random.default_rng(seed)
    seqs, labels_ = [], []
    for cls, shift in enumerate((-0.7, 0.7)):
        for _ in range(n_per):
            seqs.append(g.normal(shift, 0.6, size=(L, d)))
            labels_.append(cls)
    return np.stack(seqs), np.array(labels_)   # (B, L, d) batch

S, S_labels = make_sequences()
S_oh = np.eye(2)[S_labels]
d_model = S.shape[-1]

with params{
    Wq = 0.3 * rng.standard_normal((d_model, d_model))
    Wk = 0.3 * rng.standard_normal((d_model, d_model))
    Wv = 0.3 * rng.standard_normal((d_model, d_model))
    Wc = 0.3 * rng.standard_normal((d_model, 2))   # classifier head
    bc = np.zeros(2)
} as weights:
    # forward1 acts on ONE (L, d) sequence ($x reuses the piped value); vmap maps it
    # over the (B, L, d) batch
    forward1 = ($ |> attention($x @ Wq, $x @ Wk, $x @ Wv) |> np.mean($, axis=0) |> $ @ Wc + bc)
    batched = vmap(forward1)                       # (B, L, d) -> (B, 2)
    objective = |> S |> batched |> cross_entropy($, S_oh)
    first, last = train(weights, objective, 150, Adam(lr=0.05))
    preds = np.argmax(batched(S), axis=1)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", np.mean(preds == S_labels))

loss 0.4305 -> 0.0000
train accuracy: 1.0


## Pillar 2, continued — a Transformer encoder block

The capstone composes everything: self-attention with an output projection, a residual
connection and `layer_norm`, then a position-wise feed-forward network with its own
residual + norm — one standard Transformer encoder block, written for a single
sequence and `vmap`'d over the batch. We train it with Adam and a **cosine-decayed**
learning rate (a schedule is just a `callable(step) -> lr`).

In [10]:
def transformer_block(x, Wq, Wk, Wv, Wo, g1, beta1, W1, c1, W2, c2, g2, beta2):
    a = attention(x @ Wq, x @ Wk, x @ Wv) @ Wo   # self-attention + output proj
    x = layer_norm(x + a, g1, beta1)             # residual + norm
    ff = relu(x @ W1 + c1) @ W2 + c2             # position-wise feed-forward
    return layer_norm(x + ff, g2, beta2)         # residual + norm

d_ff = 16
with params{
    Wq = 0.3 * rng.standard_normal((d_model, d_model))
    Wk = 0.3 * rng.standard_normal((d_model, d_model))
    Wv = 0.3 * rng.standard_normal((d_model, d_model))
    Wo = 0.3 * rng.standard_normal((d_model, d_model))   # attention output proj
    g1 = np.ones(d_model)
    beta1 = np.zeros(d_model)
    W1 = 0.3 * rng.standard_normal((d_model, d_ff))      # FFN up
    c1 = np.zeros(d_ff)
    W2 = 0.3 * rng.standard_normal((d_ff, d_model))      # FFN down
    c2 = np.zeros(d_model)
    g2 = np.ones(d_model)
    beta2 = np.zeros(d_model)
    Wc = 0.3 * rng.standard_normal((d_model, 2))         # classifier head
    bc = np.zeros(2)
} as weights:
    forward1 = ($ |> transformer_block($, Wq, Wk, Wv, Wo, g1, beta1, W1, c1, W2, c2, g2, beta2)
                  |> np.mean($, axis=0) |> $ @ Wc + bc)
    batched = vmap(forward1)                       # (B, L, d) -> (B, 2)
    objective = |> S |> batched |> cross_entropy($, S_oh)
    schedule = cosine_decay(0.05, total_steps=200)
    first, last = train(weights, objective, 200, Adam(lr=schedule))
    preds = np.argmax(batched(S), axis=1)

print("loss %.4f -> %.4f  (Adam + cosine schedule)" % (first, last))
print("train accuracy:", np.mean(preds == S_labels))

loss 0.3658 -> 0.0000  (Adam + cosine schedule)
train accuracy: 1.0


## More

`vmap` is one trace level in pycograd's interpreter stack, so it composes every which
way: `vmap(grad(f))` gives the **per-sample** gradients of Pillar 1, `grad` runs straight
through the `vmap`'d sequence forwards of Pillar 2, and `vmap(vmap(f))` nests batch axes.
The bundled demos train from scratch and are gradient-checked against finite differences:

```
python -m pycograd.examples
```